In [ ]:
# ============================================================
# NOTEBOOK : Nettoyage et Pré-traitement — UAVIDS-2025
# Dataset  : Drone swarm network intrusion detection
# ============================================================

# ────────────────────────────────────────
# 1. IMPORTATION DES LIBRAIRIES
# ────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
import os

# ────────────────────────────────────────
# 2. CHARGEMENT DU DATASET
# ────────────────────────────────────────
FILE_PATH = r"C:\Drone_Attack_Similarity_Project\DATASET\UAVIDS_2025\UAVIDS-2025.csv"
df = pd.read_csv(FILE_PATH)
df_original = df.copy()  # sauvegarde de l'original

print(f" Dataset chargé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

# ────────────────────────────────────────
# 3. SUPPRESSION DES DOUBLONS
# ────────────────────────────────────────
nb_doublons = df.duplicated().sum()
print(f" détectés : {nb_doublons}")

if nb_doublons > 0:
    df = df.drop_duplicates()
    print(f"Doublons supprimés — nouveau shape : {df.shape}")
else:
    print("  Aucun doublon détecté")

# ────────────────────────────────────────
# 4. GESTION DES VALEURS MANQUANTES
# ────────────────────────────────────────
missing = df.isnull().sum()
total_missing = missing.sum()

print(f" Total valeurs manquantes : {total_missing}")

if total_missing > 0:
    num_cols = df.select_dtypes(include=[np.number]).columns
    cat_cols = df.select_dtypes(include=['object']).columns

    # Colonnes numériques → remplacement par la médiane
    for col in num_cols:
        if df[col].isnull().sum() > 0:
            median_val = df[col].median()
            df[col].fillna(median_val, inplace=True)
            print(f"  {col} — rempli par médiane ({median_val:.4f})")

    # Colonnes catégorielles → remplacement par le mode
    for col in cat_cols:
        if df[col].isnull().sum() > 0:
            mode_val = df[col].mode()[0]
            df[col].fillna(mode_val, inplace=True)
            print(f" {col} — rempli par mode ({mode_val})")
else:
    print("Aucune valeur manquante")


# ────────────────────────────────────────
# 5. SUPPRESSION DES COLONNES INUTILES
# ────────────────────────────────────────
cols_to_drop = ['FlowID']
cols_to_drop = [c for c in cols_to_drop if c in df.columns]

if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f" Colonnes supprimées : {cols_to_drop}")
else:
    print(" Aucune colonne inutile à supprimer")

print(f"   Shape après suppression : {df.shape}")


# ────────────────────────────────────────
# 6. TRAITEMENT DES OUTLIERS (IQR)
# ────────────────────────────────────────
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print(" Traitement des outliers par écrêtage (capping IQR) :\n")

nb_total_outliers = 0
for col in num_cols:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    nb_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    nb_total_outliers += nb_outliers

    if nb_outliers > 0:
        # Écrêtage : on remplace les outliers par les bornes
        df[col] = df[col].clip(lower=lower, upper=upper)
        print(f" {col} — {nb_outliers} outliers écrêtés")

print(f"\n Total outliers traités : {nb_total_outliers}")

# ────────────────────────────────────────
# 7. NORMALISATION DES FEATURES
# ────────────────────────────────────────
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

print(f"     Normalisation appliquée sur {len(num_cols)} features")
print(f"     Méthode : StandardScaler (moyenne=0, écart-type=1)")
print(f"\n   Vérification après normalisation :")
print(df[num_cols].describe().round(2).T[['mean', 'std', 'min', 'max']])

# ────────────────────────────────────────
# 8. ENCODAGE DU LABEL
# ────────────────────────────────────────
print(" Classes avant encodage :")
print(df['label'].value_counts())

le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])

print(f"\n Encodage appliqué :")
mapping = dict(zip(le.classes_, le.transform(le.classes_)))
for classe, code in mapping.items():
    print(f"   {classe} → {code}")

# ────────────────────────────────────────
# 9. VÉRIFICATION FINALE
# ────────────────────────────────────────
print("    Vérification finale :\n")
print(f"   Shape original  : {df_original.shape}")
print(f"   Shape nettoyé   : {df.shape}")
print(f"   Valeurs manquantes restantes : {df.isnull().sum().sum()}")
print(f"   Doublons restants            : {df.duplicated().sum()}")

df.head()


# ────────────────────────────────────────
# 10. SAUVEGARDE DU DATASET NETTOYÉ
# ────────────────────────────────────────
OUTPUT_PATH = r"C:\Drone_Attack_Similarity_Project\Rapport\tables\UAVIDS-2025_clean.csv"

df.to_csv(OUTPUT_PATH, index=False)
print(f"   Dataset nettoyé sauvegardé : {OUTPUT_PATH}")
print(f"   Shape final : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

 Dataset chargé : 122,171 lignes × 23 colonnes
 détectés : 0
  Aucun doublon détecté
 Total valeurs manquantes : 0
Aucune valeur manquante
 Colonnes supprimées : ['FlowID']
   Shape après suppression : (122171, 22)
 Traitement des outliers par écrêtage (capping IQR) :

 FlowDuration/s — 897 outliers écrêtés
 SrcPort — 30332 outliers écrêtés
 DstPort — 30332 outliers écrêtés
 TxPackets — 19951 outliers écrêtés
 RxPackets — 21955 outliers écrêtés
 LostPackets — 18098 outliers écrêtés
 TxBytes — 21784 outliers écrêtés
 RxBytes — 22636 outliers écrêtés
 TxPacketRate/s — 8760 outliers écrêtés
 RxPacketRate/s — 7652 outliers écrêtés
 TxByteRate/s — 8431 outliers écrêtés
 RxByteRate/s — 7335 outliers écrêtés
 MeanDelay/s — 14048 outliers écrêtés
 MeanJitter/s — 12770 outliers écrêtés
 Throughput/Kbps — 7335 outliers écrêtés
 MeanPacketSize — 30332 outliers écrêtés
 AverageHopCount — 20969 outliers écrêtés

 Total outliers traités : 283617
     Normalisation appliquée sur 18 features
     Méth